In [ ]:
import os
import re
import numpy as np
import pandas as pd
import rasterio as rio
from rasterio.merge import merge
from rasterio.plot import show
from datetime import datetime, timedelta
from pathlib import Path

In [ ]:
# GEMLST Folder
folder_path = "/media/sfm/Local Data/SimonKleiner/GEMLST/Gapfilled"

naming_con = "GEMLST_MODIS_YYYYMMDD.tiff" 

files = sorted(Path(folder_path).glob("GEMLST_????????.tiff"))

SOURCE_BITS = {
    "ERA5" : 0,
    "Terra_day": 1,
    "Terra_night": 2,
    "Aqua_day": 4,
    "Aqua_night": 8,
    "VIIRS_day": 16,
    "VIIRS_night": 32,
    "JAXA_A": 64,
    "JAXA_B": 128
}

# print(files)


### DATASET CREATION: 
The following cells create a dataset with pixel counts for each sensor from all GEMLST images. 
Skip this, if datasets qa_band_source_counts_raw and qa_band_source_counts_aggregated already exist, as this step takes a while even in parallel processing. 

In [ ]:
def count_sources_in_qa_band(file):
    """Count valid pixels represented by each source in QA band 2."""
    with rio.open(file) as src:
        qa = src.read(2)

    valid = np.isfinite(qa)

    # if nodata is not None and np.isfinite(nodata):
    #     valid &= qa != nodata

    # QA values are bit fields; invalid pixels are set to zero.
    qa_int = np.zeros(qa.shape, dtype=np.uint64)
    qa_int[valid] = qa[valid].astype(np.uint64)

    counts = {
    "ERA5": int(np.count_nonzero(valid & (qa_int == 0)))
    }

    counts.update({
        source: int(np.count_nonzero(
            valid & ((qa_int & bit) != 0)
        ))
        for source, bit in SOURCE_BITS.items()
        if source != "ERA5"
    })

    return counts

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp


def process_file(file):
    """Count QA-band sources for one raster and return a DataFrame row."""
    date_str = file.stem.split("_")[1]
    date = datetime.strptime(date_str, "%Y%m%d").date()

    row = {
        "Date": date,
    }
    row.update(count_sources_in_qa_band(file))
    return row


if not files:
    raise ValueError(f"No input files found in {folder_path}")

# Raster files are independent, so distribute one file per process.
with ProcessPoolExecutor(
    max_workers=os.cpu_count(),
    mp_context=mp.get_context("fork"),
) as executor:
    rows = list(executor.map(process_file, sorted(files)))

result_df = pd.DataFrame(rows).sort_values("Date").reset_index(drop=True)

result_df["total_count"] = result_df[list(SOURCE_BITS.keys())].sum(axis=1)

result_df.to_csv("qa_band_source_counts_raw.csv", index=False)

print(result_df)
# print(result_df)

In [ ]:
# Load the CSV file into a DataFrame
df = pd.read_csv("qa_band_source_counts_raw.csv")

df["Date"] = pd.to_datetime(df["Date"])

columns = list(SOURCE_BITS.keys()) + ["total_count"]

agg_df = (
    df.groupby(df["Date"].dt.year)[columns]
      .sum()
)

agg_df.index.name = "Year"

agg_df.to_csv("qa_band_source_counts_aggregated.csv", index=True)

### PLOTTING SECTION

In [ ]:
# Merge the day and night obs into one column for each source for plotting

plot_df = pd.read_csv("qa_band_source_counts_aggregated.csv")
plot_df["MODIS/Terra"] = plot_df["Terra_day"] + plot_df["Terra_night"]
plot_df["MODIS/Aqua"] = plot_df["Aqua_day"] + plot_df["Aqua_night"]
plot_df["VIIRS"] = plot_df["VIIRS_day"] + plot_df["VIIRS_night"]
plot_df["SGLI"] = plot_df["JAXA_A"] + plot_df["JAXA_B"]

print(plot_df)


### Stats

In [29]:
# Stats
total_valid = 782335525

plot_df['total_sat'] = plot_df['total_count'] - plot_df['ERA5']

sat_2001 = plot_df[plot_df['Year'] == 2001][('MODIS/Terra')]
mean_from_2018_2025 = plot_df[plot_df['Year'] >= 2018][('total_sat')].mean()

print(mean_from_2018_2025/sat_2001)

print('MODIS/Terra', plot_df[plot_df['Year'] >= 2001]['MODIS/Terra'].mean())
print('MODIS/Aqua', plot_df[plot_df['Year'] >= 2002]['MODIS/Aqua'].mean())
print('VIIRS', plot_df[plot_df['Year'] >= 2012]['VIIRS'].mean())
print('SGLI', plot_df[plot_df['Year'] >= 2018]['SGLI'].mean())

gaps1 = plot_df[(plot_df['Year'] >= 2001) & (plot_df['Year'] < 2012)]['ERA5'].mean()
gaps2 = plot_df[(plot_df['Year'] >= 2018) & (plot_df['Year'] < 2025)]['ERA5'].mean()

print(gaps2/gaps1)

plot_df['gaps_pct'] = plot_df['ERA5'] / total_valid * 100

gaps_pct1 = plot_df[(plot_df['Year'] >= 2001) & (plot_df['Year'] < 2012)]['gaps_pct'].mean()
gaps_pct2 = plot_df[(plot_df['Year'] >= 2018) & (plot_df['Year'] < 2025)]['gaps_pct'].mean()

print('gaps 2001-2011', gaps_pct1)
print('gaps 2018-2025', gaps_pct2)

with rio.open("/home/sirsimsius/Hentet/GEMLST_20251231.tiff") as src:
    data = src.read(2)
    valid_pixels_pr_img = np.isfinite(data).sum()
    print(f"Valid pixels per single GEMLST img: {valid_pixels_pr_img}")

print('Extra square kilometers mapped:', (valid_pixels_pr_img * (1-gaps_pct2/100))-(valid_pixels_pr_img * (1-gaps_pct1/100)))




1    3.485883
Name: MODIS/Terra, dtype: float64
MODIS/Terra 638342748.52
MODIS/Aqua 582240498.125
VIIRS 710912898.1428572
SGLI 275499712.125
0.4729340626240244
gaps 2001-2011 24.63365720923199
gaps 2018-2025 11.650095581249673
Valid pixels per single GEMLST img: 2143385
Extra square kilometers mapped: 278287.7123999286


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import cmcrameri.cm as cmc
import cmcrameri

# sns.set_style("whitegrid")

relevant_columns = ["MODIS/Terra", "MODIS/Aqua", "VIIRS", "SGLI", "ERA5"]
# n_colors = len(relevant_columns)
# color = [cmc.batlow(i / n_colors) for i in range(n_colors)]


palette = np.asarray(cmc.batlowW.colors)
# Select well-separated colours across the complete palette
selected = palette[np.linspace(0, len(palette)-30, len(relevant_columns), dtype=int)]
# Rearrange the selected colours
order = [0, 1, 2, 3, 4]
colors = selected[order]

# Change the colour palette to cmcrameri "batlowS"
fig, (ax1, ax2, ax3) = plt.subplots(
    1, 3, 
    figsize=(15, 4),
    dpi=300,
    gridspec_kw={'width_ratios': [6, 3, 3], 'wspace': 0},
    constrained_layout=True
)
plot_df.plot(x="Year", y=relevant_columns, kind="bar", stacked=True, ax=ax1, color=colors, width=0.8, edgecolor="black", linewidth=0.5)
ax1.axhline(
    y=782335525,
    color="black",
    linestyle="--",
    linewidth=1,
    label="Valid pixels",
)
ax1.legend()
ax1.set_xlabel("Year")
ax1.set_ylabel("N observations (pixels * $10⁹$)")
ax1.tick_params(axis="x", rotation=45)

# Create a cake diagram that sums all years and shows the relative contribution of each source to the total valid pixels.
total_counts = plot_df[relevant_columns].sum()
ax2.pie(total_counts, autopct='%1.1f%%', textprops={'fontsize': 10, 'color': 'black'}, pctdistance=1.25, colors=colors, startangle=90, counterclock=False, radius=0.8)

# Add a heatmap of the geotiff inkl. a colorbar to the right of the pie chart
# Load the first raster file to get the data for the heatmap
qa_file = "/home/sirsimsius/Dokumente/Arbeit/KU/GIS/Validation/QA_band_zeros_total.tiff"
with rio.open(qa_file) as src:
    qa_data = src.read(1)
    # # Set invalid pixels to NaN for better visualization
    # valid = np.isfinite(qa_data)
    # qa_data[valid] = qa_data[valid].astype(np.int32)

# Plot qa_data using matplotlib's imshow
    im = ax3.imshow(qa_data, cmap='cmc.batlow', interpolation='nearest')
    cbar = plt.colorbar(im, ax=ax3, fraction=0.046, pad=0.04)
    cbar.set_label('QA Band Values', rotation=270, labelpad=15)
    ax3.axis('off')


# Export as pdf and png
# plt.savefig("qa_band_source_counts.pdf", dpi=300, bbox_inches='tight')
# plt.savefig("qa_band_source_counts.png", dpi=300, bbox_inches='tight')

plt.tight_layout()
plt.show()

